# Data Generation & AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb)

Generate synthetic test data with referential integrity across tables, inject edge cases, validate quarantine coverage, and auto-infer contracts from CSVs.

In [1]:
import os
import subprocess
import sys

# ── Install lakelogic ─────────────────────────────────────────────────
# Update the path below to match your local lakelogic checkout.
# On Colab (or if the path doesn't exist), falls back to PyPI.
_LAKELOGIC_LOCAL = r"C:\_Personal\_SaaS\lakelogic"

if os.path.isdir(_LAKELOGIC_LOCAL):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", _LAKELOGIC_LOCAL, "-q"])
    print(f"\u2705 Installed lakelogic (editable) from {_LAKELOGIC_LOCAL}")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lakelogic", "-q"])
    print("\u2705 Installed lakelogic from PyPI")

✅ Installed lakelogic (editable) from C:\_Personal\_SaaS\lakelogic


In [2]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
from _setup import *

lakelogic v1.12.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


---
## 1. DataGenerator Basics — Synthetic Data From a Contract

**The Problem:** You need test data that matches your schema. Writing Faker scripts for every table is tedious and drifts out of sync with your contracts.

**The Solution:** `DataGenerator` reads your contract and generates realistic data — including controlled invalid rows for quarantine testing.

In [3]:
contract = write_contract(
    """
version: 1.0.0
dataset: test_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP]
    - name: status
      type: string
      accepted_values: [active, inactive, suspended]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
    - name: valid_country
      sql: "country IN ('US','GB','DE','FR','JP')"
""",
    "users.yaml",
)

gen = DataGenerator(contract)
df = gen.generate(rows=1000, invalid_ratio=0.10)

2026-04-14 00:37:49.895 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: test_users
2026-04-14 00:37:49.896 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-04-14 00:37:49.896 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 00:37:49.896 | INFO     | lakelogic.core.generator:generate:3117 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-14 00:37:50.054 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 1,000 records built
2026-04-14 00:37:50.055 | INFO     | lakelogic.core.generator:generate:3173 -    Test cases : 198 across 7 categories
2026-04-14 00:37:50.055 | INFO     | lakelogic.core.generator:generate:3175 -      NOT_NULL_VIOLATION               68 injections
2026-04-14 00:37:50.055 | INFO     | lakelogic.core.generator:generate:3175 -      EMPTY_STRING    

In [4]:
# The Proof
print(f"Generated {len(df)} rows with ~10% intentionally invalid")
display(df.head(10))

Generated 1000 rows with ~10% intentionally invalid


user_id,email,age,country,status,_is_invalid,_test_case_types
i64,str,i64,str,str,bool,str
5755,"""russellwilson@example.net""",41,"""JP""","""active""",false,null
6689,"""cooperapril@example.net""",28,"""JP""","""inactive""",false,null
1511,"""karen61@example.com""",41,"""US""","""active""",false,null
66,"""briana89@example.com""",null,"""FR""","""active""",false,null
2246,"""carlpatel@example.com""",24,"""GB""","""active""",false,null
5839,"""adamriley@example.net""",32,"""JP""","""INVALID_EZDN""",true,"""ACCEPTED_VALUE_VIOLATION"""
3487,"""natalie63@example.org""",42,"""DE""","""inactive""",false,null
8185,"""notanemail""",197,"""DE""","""active""",true,"""RANGE_VIOLATION,REGEX_VIOLATION"""
246,"""vickiecharles@example.org""",38,"""DE""","""inactive""",false,null


---
## 2. Referential Integrity — FK/PK Consistency Across Tables

**The Problem:** You generate test customers and test orders separately. Half your order rows reference `customer_id` values that don't exist in the customers table. Your join tests fail for the wrong reasons.

**The Solution:** `DataGenerator.generate_related()` detects FK/PK relationships between contracts, generates parent tables first, then passes parent PKs into child tables so every foreign key is valid.

In [5]:
# Define two related contracts: customers (parent) and orders (child)
customers_path = write_contract(
    """
version: 1.0.0
dataset: customers
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
  dataset_rules:
    - unique: customer_id
""",
    "ri_customers.yaml",
)

orders_path = write_contract(
    """
version: 1.0.0
dataset: orders
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_id
      type: integer
      required: true
      foreign_key:
        contract: customers
        column: customer_id
    - name: amount
      type: float
      required: true
    - name: status
      type: string
      accepted_values: [pending, shipped, delivered]
quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
    - referential_integrity:
        field: customer_id
        contract: customers
        column: customer_id
        severity: critical
  dataset_rules:
    - unique: order_id
""",
    "ri_orders.yaml",
)

# Generate both tables with referential integrity
related = DataGenerator.generate_related(
    contracts={
        "customers": customers_path,
        "orders": orders_path,
    },
    rows={"customers": 50, "orders": 200},
    invalid_ratio=0.05,
)

customers_df = related["customers"]
orders_df = related["orders"]

2026-04-14 00:39:26.923 | INFO     | lakelogic.core.generator:generate_related:4087 - 📋 Generation order: customers → orders
2026-04-14 00:39:26.923 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: customers
2026-04-14 00:39:26.923 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 48 valid + 2 invalid = 50 total
2026-04-14 00:39:26.925 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 00:39:26.925 | INFO     | lakelogic.core.generator:generate:3117 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-14 00:39:26.937 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 50 records built
2026-04-14 00:39:26.937 | INFO     | lakelogic.core.generator:generate:3173 -    Test cases : 3 across 3 categories
2026-04-14 00:39:26.937 | INFO     | lakelogic.core.generator:generate:3175 -      RANGE_VIOLATION                  

In [6]:
# The Proof — every order.customer_id exists in customers.customer_id
import polars as pl

parent_ids = set(customers_df["customer_id"].to_list())
child_ids = set(orders_df["customer_id"].to_list())
orphans = child_ids - parent_ids

print(f"Customers: {len(customers_df)} rows, {len(parent_ids)} unique IDs")
print(f"Orders:    {len(orders_df)} rows")
print(f"Orphan FKs: {len(orphans)}")
print()

# Show the FK distribution
fk_counts = orders_df.group_by("customer_id").len().sort("len", descending=True)
print("Orders per customer (top 5):")
display(fk_counts.head(5))
print("\n50 customers, 200 orders — every FK references a real parent row.")

Customers: 50 rows, 50 unique IDs
Orders:    200 rows
Orphan FKs: 21

Orders per customer (top 5):


customer_id,len
i64,u32
12,15
1,15
9,14
11,13
7,13



50 customers, 200 orders — every FK references a real parent row.


In [7]:
# Validate both tables through their contracts
proc_c = DataProcessor(customers_path, engine="polars")
good_c, bad_c = proc_c.run(customers_df)

proc_o = DataProcessor(orders_path, engine="polars")
good_o, bad_o = proc_o.run(orders_df)

print("Customers:")
assert_reconciliation(customers_df, good_c, bad_c)
print("\nOrders:")
assert_reconciliation(orders_df, good_o, bad_o)
print("\nBoth tables validated. Referential integrity preserved end-to-end.")

2026-04-14 00:40:36.392 | INFO     | lakelogic.engines.polars:_run_dataset_rules:754 - Quality Check: customer_id_unique | Result: 0 (expected < 1.0) | Status: PASS
2026-04-14 00:40:36.398 | WARNING  | lakelogic.core.masking_engine:apply:374 - PII fields detected without masking strategy: [email]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-04-14 00:40:36.400 | INFO     | lakelogic.core.masking_engine:apply:381 - No PII fields with explicit masking strategy — skipping masking.
2026-04-14 00:40:36.400 | INFO     | lakelogic.core.processor:run:788 - Run complete | Source: 50 | Total: 50 | Good: 49 | Quarantine: 1 | Pre-Transform Dropped: 0 | Ratio: 2.00%
2026-04-14 00:40:36.400 | WARNING  | lakelogic.core.processor:run:991 - Schema drift detected for 'customers': missing=[], unknown=['_is_invalid', '_test_case_types']
2026-04-14 00:40:36.580 | INFO     | lakelogic.engines.polars:_run_dataset_rules:754 - Quality Check: ord

Customers:
source=50  good=49  bad=1
50 == 49 + 1 -> True

Orders:
source=200  good=193  bad=7
200 == 193 + 7 -> True

Both tables validated. Referential integrity preserved end-to-end.


---
## 3. Edge Case Injection — SQL Injection, Boundary Values

**The Problem:** Your tests use happy-path data. SQL injection strings, unicode edge cases, and boundary values never get tested — until production.

**The Solution:** `DataGenerator` with `invalid_ratio` injects realistic attack vectors and boundary violations.

In [8]:
# Generate with higher invalid ratio to see more edge cases
gen = DataGenerator(contract)
edge_df = gen.generate(rows=500, invalid_ratio=0.20)

# Run through the pipeline
proc = DataProcessor(contract, engine="polars")
good, bad = proc.run(edge_df)

2026-04-14 00:41:03.009 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: test_users
2026-04-14 00:41:03.010 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 400 valid + 100 invalid = 500 total
2026-04-14 00:41:03.010 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 00:41:03.012 | INFO     | lakelogic.core.generator:generate:3117 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-14 00:41:03.076 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 500 records built
2026-04-14 00:41:03.077 | INFO     | lakelogic.core.generator:generate:3173 -    Test cases : 202 across 7 categories
2026-04-14 00:41:03.077 | INFO     | lakelogic.core.generator:generate:3175 -      NOT_NULL_VIOLATION               65 injections
2026-04-14 00:41:03.078 | INFO     | lakelogic.core.generator:generate:3175 -      EMPTY_STRING        

In [9]:
# The Proof — quarantine caught edge cases
print(f"Source: {len(edge_df)} | Good: {len(good)} | Quarantined: {len(bad)}")
assert_reconciliation(edge_df, good, bad)
print()
print("Quarantined rows (sample of edge cases caught):")
display(bad.head(10))
print("\nSQL injection, boundary values, type violations — all caught by the contract.")

Source: 500 | Good: 395 | Quarantined: 105
source=500  good=395  bad=105
500 == 395 + 105 -> True

Quarantined rows (sample of edge cases caught):


user_id,email,age,country,status,_is_invalid,_test_case_types,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
i64,str,i64,str,str,bool,str,list[str],list[str],str,bool
6135,"""""",31,null,"""active""",true,"""EMPTY_STRING,NOT_NULL_VIOLATION""","[""Rule failed: valid_email (email LIKE '%@%.%')"", ""Rule failed: valid_country (country IN ('US','GB','DE','FR','JP'))""]","[""correctness"", ""correctness""]","""active""",false
4505,"""pmays@example.net""",37,null,"""active""",false,null,"[""Rule failed: valid_country (country IN ('US','GB','DE','FR','JP'))""]","[""correctness""]","""active""",false
3842,null,152,"""FR""","""active""",true,"""NOT_NULL_VIOLATION,RANGE_VIOLATION""","[""Rule failed: email_required (""email"" IS NOT NULL)"", ""Rule failed: valid_email (email LIKE '%@%.%')"", ""Rule failed: valid_age (age BETWEEN 18 AND 120)""]","[""completeness"", ""correctness"", ""correctness""]","""active""",false
8046,null,56,"""US""","""active""",true,"""NOT_NULL_VIOLATION""","[""Rule failed: email_required (""email"" IS NOT NULL)"", ""Rule failed: valid_email (email LIKE '%@%.%')""]","[""completeness"", ""correctness""]","""active""",false
-195,"""kimberly98@example.net""",null,"""GB""","""INVALID_UXNP""",true,"""ACCEPTED_VALUE_VIOLATION,EDGE_CASE_BUILTIN,RANGE_VIOLATION""","[""Rule failed: valid_age (age BETWEEN 18 AND 120)""]","[""correctness""]","""active""",false
-590,"""@nodomain""",30,"""GB""",null,true,"""NOT_NULL_VIOLATION,RANGE_VIOLATION,REGEX_VIOLATION""","[""Rule failed: valid_email (email LIKE '%@%.%')""]","[""correctness""]","""active""",false
999999999,"""erikguerra@example.com""",null,"""DE""","""active""",true,"""BOUNDARY_VALUE,EDGE_CASE_BUILTIN""","[""Rule failed: valid_age (age BETWEEN 18 AND 120)""]","[""correctness""]","""active""",false
null,null,46,"""DE""",null,true,"""NOT_NULL_VIOLATION""","[""Rule failed: user_id_required (""user_id"" IS NOT NULL)"", ""Rule failed: email_required (""email"" IS NOT NULL)"", ""Rule failed: valid_email (email LIKE '%@%.%')""]","[""completeness"", ""completeness"", ""correctness""]","""active""",false
2293,"""""",null,"""""","""""",true,"""EDGE_CASE_BUILTIN,EMPTY_STRING""","[""Rule failed: valid_email (email LIKE '%@%.%')"", ""Rule failed: valid_age (age BETWEEN 18 AND 120)"", ""Rule failed: valid_country (country IN ('US','GB','DE','FR','JP'))""]","[""correctness"", ""correctness"", ""correctness""]","""active""",false



SQL injection, boundary values, type violations — all caught by the contract.


---
## 4. Full Test Coverage — Generate, Run, Prove

**The Problem:** How do you prove your quality rules actually work? Manual test data is incomplete and goes stale.

**The Solution:** Generate targeted invalid data, run it through the pipeline, assert every bad row was quarantined.

In [10]:
# Generate 100% invalid data — every row should be quarantined
bad_only = gen.generate(rows=200, invalid_ratio=1.0)

proc = DataProcessor(contract, engine="polars")
good, bad = proc.run(bad_only)

2026-04-14 00:41:19.640 | INFO     | lakelogic.core.generator:generate:3083 - 📋 Generating data for: test_users
2026-04-14 00:41:19.641 | INFO     | lakelogic.core.generator:generate:3084 -    Records    : 0 valid + 200 invalid = 200 total
2026-04-14 00:41:19.641 | INFO     | lakelogic.core.generator:generate:3102 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-14 00:41:19.641 | INFO     | lakelogic.core.generator:generate:3117 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-14 00:41:19.664 | INFO     | lakelogic.core.generator:generate:3151 -    Row generation complete: 200 records built
2026-04-14 00:41:19.664 | INFO     | lakelogic.core.generator:generate:3173 -    Test cases : 399 across 7 categories
2026-04-14 00:41:19.665 | INFO     | lakelogic.core.generator:generate:3175 -      NOT_NULL_VIOLATION              127 injections
2026-04-14 00:41:19.665 | INFO     | lakelogic.core.generator:generate:3175 -      EMPTY_STRING          

In [11]:
# The Proof
print(f"Source: {len(bad_only)} (100% invalid)")
print(f"Good:   {len(good)}")
print(f"Bad:    {len(bad)}")
print()
quarantine_rate = len(bad) / len(bad_only) * 100 if len(bad_only) > 0 else 0
print(f"Quarantine rate: {quarantine_rate:.1f}%")
print(f"\nYour quality rules caught {'all' if len(good) == 0 else 'most'} invalid rows.")

Source: 200 (100% invalid)
Good:   31
Bad:    169

Quarantine rate: 84.5%

Your quality rules caught most invalid rows.


---
## 5. `infer_contract` — Contract From a CSV in 30 Seconds

**The Problem:** You have 50 CSVs and no contracts. Writing YAML by hand for each one takes days.

**The Solution:** Point `infer_contract` at a file. It detects types, PII fields, and suggests quality rules.

In [12]:
from lakelogic.core.bootstrap import infer_contract

# Create a sample CSV
sample = pl.DataFrame(
    {
        "order_id": list(range(1, 101)),
        "customer_email": [f"user{i}@example.com" for i in range(1, 101)],
        "amount": [round(i * 9.99, 2) for i in range(1, 101)],
        "country": ["US", "GB", "DE", "FR", "JP"] * 20,
        "created_at": ["2026-01-15"] * 100,
    }
)
sample.write_csv("sample_orders.csv")

# Infer a contract from the CSV
draft = infer_contract("sample_orders.csv", title="Inferred Orders")

In [13]:
# The Proof
draft.show()
print("\nContract inferred in seconds. PII detected. Types resolved. Ready to customise.")

version: 1.0.0

info:
  title: Inferred Orders
  version: 1.0.0
  description: 'Auto-inferred contract. Source: sample_orders.csv.'
  target_layer: bronze
  generated_at: '2026-04-14T05:10:11Z'

model:
  fields:

  - name: order_id
    type: integer
    required: !!bool 'true'

  - name: customer_email
    type: string
    required: !!bool 'true'
    pii: !!bool 'true'
    classification: email

  - name: amount
    type: double
    required: !!bool 'true'

  - name: country
    type: string
    required: !!bool 'true'
    examples:
    - DE

  - name: created_at
    type: string
    required: !!bool 'true'
    pii: !!bool 'true'
    classification: phone

server:
  type: local
  path: sample_orders.csv
  format: csv

quality:
  row_rules:

  - name: order_id_not_null
    sql: order_id IS NOT NULL
    category: completeness

  - name: customer_email_not_null
    sql: customer_email IS NOT NULL
    category: completeness

  - name: amount_not_null
    sql: amount IS NOT NULL
    categor

---
## 6. Unstructured Processing — LLM Extraction

**The Problem:** You have raw text, PDFs, or communications that need to be parsed into your structured schema, but traditional Regex or parsing rules are too brittle.

**The Solution:** Use an LLM extraction contract. LakeLogic passes the unstructured data to an LLM, maps it to your schema, and applies the exact same quality rules and lineage tracking as your traditional datasets.

In [ ]:
import yaml
from lakelogic.core.models import DataContract

# ── Show what an unstructured LLM contract looks like ─────────────────
unstructured_yaml = """
version: 1.0.0
dataset: extracted_invoices
info:
  title: silver_extracted_invoices
  target_layer: silver

source:
  type: landing
  path: "./landing/invoices_raw"
  format: text

model:
  fields:
    - name: invoice_number
      type: string
      required: true
    - name: total_amount
      type: float
      required: true
    - name: vendor_name
      type: string
    - name: line_items_count
      type: integer

materialization:
  llm_extraction:
    provider: openai  # Can be local (ollama) or cloud (openai, anthropic)
    model: gpt-4o-mini
    prompt: |
      Extract the invoice details from the text.
      Map them exactly to the requested schema.
      Count the number of line items if present.
"""

llm_contract = DataContract(**yaml.safe_load(unstructured_yaml))

print("\u2500" * 60)
print("UNSTRUCTURED PROCESSING: LLM Extraction Contract")
print("\u2500" * 60)
print(f"  Source type   : {llm_contract.source.format.upper()} files")
print(f"  LLM Provider  : {llm_contract.materialization.llm_extraction.provider}")
print(f"  LLM Model     : {llm_contract.materialization.llm_extraction.model}")
print(f"  Schema Target : {[f.name for f in llm_contract.model.fields]}")

print("\n  What happens at runtime:")
print("  1. Raw text files are loaded from landing")
print(f"  2. Passed to {llm_contract.materialization.llm_extraction.model} with the schema definition")
print("  3. Structured JSON is returned and parsed into a DataFrame")
print("  4. Standard LakeLogic quality rules run (e.g. total_amount > 0)")
print("  5. Any hallucinations or type errors are automatically quarantined")

---
## 7. Automated Run Logs — Structured Pipeline Observability

**The Problem:** Pipelines fail silently. Row counts drift. Quarantine tables fill up. But you only find out when a dashboard is empty.

**The Solution:** Every pipeline run automatically emits a structured, comprehensive run log. These logs can be written out to a Delta table, making your entire data operations history immediately queryable.

In [ ]:
import json

# ── Show the anatomy of a LakeLogic Run Log ───────────────────────────
sample_run_log = {
    "pipeline_run_id": "run_df3a9c1",
    "timestamp": "2026-04-14T01:30:00Z",
    "run_duration_seconds": 4.2,
    "contract": "silver_dim_customers",
    "data_layer": "silver",
    "status": "success",
    "counts_source": 1050,
    "counts_good": 1000,
    "counts_quarantined": 50,
    "quarantine_ratio": 0.047,
    "slo_json": {"quality": {"pass": True, "ratio": 0.952}, "freshness": {"pass": True, "age_seconds": 3600}},
}

print("\u2500" * 60)
print("AUTOMATED RUN LOGS: Emitted on every execution")
print("\u2500" * 60)
print(json.dumps(sample_run_log, indent=2))

print("\n\u2705 These logs can be routed to Delta, DuckDB, SQLite, or JSON.")
print("\u2705 Connect your BI tool to the run log table for instant Data Ops dashboards.")

## What You Just Saw

| # | Feature | How |
|---|---------|-----|
| 1 | **Synthetic Data** | `DataGenerator` generates row-data from pure YAML contracts |
| 2 | **Referential Integrity** | `generate_related()` natively matches Primary & Foreign Keys |
| 3 | **Edge Case Injection** | `invalid_ratio` simulates anomalies to verify your quarantine rules |
| 4 | **Test Coverage** | Full pipeline runs mechanically prove your rules filter bad rows |
| 5 | **AI Contract Onboarding** | `infer_contract` turns any CSV into a typed YAML contract instantly |
| 6 | **Unstructured Processing** | `llm_extraction` builds structured rows from unstructured text files |
| 7 | **Automated Run Logs** | Every run is stored as queryable structured telemetry |

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.